In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.metrics import Precision, Recall
from sklearn.model_selection import ParameterGrid
import time
import json

print("Phase 6: Model Optimization & Hyperparameter Tuning")
print("=" * 60)
print("Task 1: Loading data and setting up environment")

# Load preprocessed data
print("Loading training data...")
embedding_matrix = np.load('embedding_matrix.npy')
X_train_words = np.load('X_train_preprocessed.npy')
X_train_chars = np.load('train_char_sequences.npy')
X_train_struct = np.load('train_sql_features.npy')
y_train = np.load('y_train_labels.npy')

X_val_words = np.load('X_val_preprocessed.npy')
X_val_chars = np.load('val_char_sequences.npy')
X_val_struct = np.load('val_sql_features.npy')
y_val = np.load('y_val_labels.npy')

print("Loading test data...")
X_test_words = np.load('X_test_preprocessed.npy')
X_test_chars = np.load('test_char_sequences.npy')
X_test_struct = np.load('test_sql_features.npy')
y_test = np.load('y_test_labels.npy')

# Prepare data inputs
train_inputs = [X_train_words, X_train_chars, X_train_struct]
val_inputs = [X_val_words, X_val_chars, X_val_struct]
test_inputs = [X_test_words, X_test_chars, X_test_struct]

# Load Phase 5 best model as baseline
print("Loading Phase 5 baseline model...")
baseline_model = load_model('best_model_phase5.h5')

# Class weights from Phase 1
class_weights = {
    0: 1.4805809746754628,
    1: 0.7549508979436819
}

print("Data loaded successfully:")
print(f"Training samples: {X_train_words.shape[0]:,}")
print(f"Validation samples: {X_val_words.shape[0]:,}")
print(f"Test samples: {X_test_words.shape[0]:,}")
print(f"Embedding matrix shape: {embedding_matrix.shape}")

# Evaluate baseline model performance
print("\nEvaluating Phase 5 baseline performance...")
baseline_val_loss, baseline_val_acc, baseline_val_prec, baseline_val_rec = baseline_model.evaluate(
    val_inputs, y_val, verbose=0
)

print(f"Baseline Model Performance:")
print(f"- Validation Accuracy: {baseline_val_acc:.4f}")
print(f"- Validation Loss: {baseline_val_loss:.4f}")
print(f"- Validation Precision: {baseline_val_prec:.4f}")
print(f"- Validation Recall: {baseline_val_rec:.4f}")

print("\nPhase 6 environment setup completed")
print("Ready for hyperparameter optimization")


Phase 6: Model Optimization & Hyperparameter Tuning
Task 1: Loading data and setting up environment
Loading training data...
Loading test data...
Loading Phase 5 baseline model...
Data loaded successfully:
Training samples: 55,658
Validation samples: 11,979
Test samples: 11,934
Embedding matrix shape: (20000, 300)

Evaluating Phase 5 baseline performance...
Baseline Model Performance:
- Validation Accuracy: 0.9942
- Validation Loss: 0.0243
- Validation Precision: 0.9975
- Validation Recall: 0.9937

Phase 6 environment setup completed
Ready for hyperparameter optimization


In [2]:
print("Task 2: Creating Parameterized Model Architecture")
print("=" * 50)

# Architecture constants
max_word_length = 75
max_char_length = 300
char_vocab_size = 72
char_embedding_dim = 50
struct_feature_dim = 135
filter_sizes = [3, 4, 5]
num_filters = 128

def build_optimized_cnn_model(embedding_matrix, 
                             dropout_rate=0.5, 
                             l2_reg=1e-5, 
                             dense_units_1=256, 
                             dense_units_2=64,
                             num_conv_filters=128):
    """
    Build parameterized CNN model for hyperparameter optimization
    
    Parameters:
    - embedding_matrix: Pre-trained embeddings
    - dropout_rate: Dropout probability (0.3-0.7)
    - l2_reg: L2 regularization strength (1e-6 to 1e-4)
    - dense_units_1: First dense layer units (128-512)
    - dense_units_2: Second dense layer units (32-128)
    - num_conv_filters: Number of CNN filters (64-256)
    """
    
    # Word branch
    word_input = Input(shape=(max_word_length,), name='word_input')
    word_embedding = Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        input_length=max_word_length,
        trainable=True,
        name='word_embedding'
    )(word_input)
    
    word_convs = []
    for filter_size in filter_sizes:
        conv = Conv1D(num_conv_filters, filter_size, activation='relu')(word_embedding)
        pool = MaxPooling1D(pool_size=max_word_length - filter_size + 1)(conv)
        flatten = Flatten()(pool)
        word_convs.append(flatten)
    word_output = concatenate(word_convs)
    
    # Character branch
    char_input = Input(shape=(max_char_length,), name='char_input')
    char_embedding = Embedding(
        input_dim=char_vocab_size,
        output_dim=char_embedding_dim,
        input_length=max_char_length,
        trainable=True,
        name='char_embedding'
    )(char_input)
    
    char_convs = []
    for filter_size in filter_sizes:
        conv = Conv1D(num_conv_filters, filter_size, activation='relu')(char_embedding)
        pool = MaxPooling1D(pool_size=max_char_length - filter_size + 1)(conv)
        flatten = Flatten()(pool)
        char_convs.append(flatten)
    char_output = concatenate(char_convs)
    
    # Structural branch
    struct_input = Input(shape=(struct_feature_dim,), name='struct_input')
    struct_dense1 = Dense(256, activation='relu')(struct_input)
    struct_dropout1 = Dropout(dropout_rate)(struct_dense1)
    struct_dense2 = Dense(128, activation='relu')(struct_dropout1)
    
    # Fusion layers with parameterized architecture
    combined = concatenate([word_output, char_output, struct_dense2])
    combined = BatchNormalization()(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(dense_units_1, activation='relu', 
                    kernel_regularizer=regularizers.l2(l2_reg))(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(dense_units_2, activation='relu', 
                    kernel_regularizer=regularizers.l2(l2_reg))(combined)
    output = Dense(1, activation='sigmoid')(combined)
    
    model = Model(inputs=[word_input, char_input, struct_input], outputs=output)
    return model

# Test architecture variations
print("Testing parameterized architecture variations:")

test_configs = [
    {'dropout_rate': 0.3, 'l2_reg': 1e-6, 'dense_units_1': 128, 'dense_units_2': 32},
    {'dropout_rate': 0.5, 'l2_reg': 1e-5, 'dense_units_1': 256, 'dense_units_2': 64},  # Baseline
    {'dropout_rate': 0.7, 'l2_reg': 1e-4, 'dense_units_1': 512, 'dense_units_2': 128}
]

for i, config in enumerate(test_configs, 1):
    model_test = build_optimized_cnn_model(embedding_matrix, **config)
    params = model_test.count_params()
    print(f"Config {i}: {params:,} parameters - {config}")

print(f"\nBaseline model parameters: {baseline_model.count_params():,}")
print("Parameterized architecture ready for grid search")
print("Task 2 completed successfully")


Task 2: Creating Parameterized Model Architecture
Testing parameterized architecture variations:
Config 1: 6,732,241 parameters - {'dropout_rate': 0.3, 'l2_reg': 1e-06, 'dense_units_1': 128, 'dense_units_2': 32}
Config 2: 6,859,409 parameters - {'dropout_rate': 0.5, 'l2_reg': 1e-05, 'dense_units_1': 256, 'dense_units_2': 64}
Config 3: 7,138,321 parameters - {'dropout_rate': 0.7, 'l2_reg': 0.0001, 'dense_units_1': 512, 'dense_units_2': 128}

Baseline model parameters: 6,859,409
Parameterized architecture ready for grid search
Task 2 completed successfully


In [3]:
print("Task 3: Hyperparameter Grid Search Optimization")
print("=" * 55)

# Define hyperparameter search space
param_grid = {
    'optimizer': ['adam', 'rmsprop'],
    'learning_rate': [0.0005, 0.001, 0.002],
    'batch_size': [16, 32, 64],
    'dropout_rate': [0.3, 0.5, 0.7],
    'l2_reg': [1e-6, 1e-5, 1e-4],
    'dense_units_1': [128, 256, 512],
    'dense_units_2': [32, 64, 128]
}

print("Hyperparameter search space:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

# Calculate total combinations
total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nTotal possible combinations: {total_combinations:,}")

# Recommended approach: 25 combinations with selective detailed logging
max_search_combinations = 25
detailed_combinations = 5  # Detailed logging for first 5
print(f"Testing {max_search_combinations} strategic combinations")
print(f"Detailed epoch logging for first {detailed_combinations} combinations")
print(f"Summary logging for remaining {max_search_combinations - detailed_combinations} combinations")

# Generate strategic parameter combinations
np.random.seed(42)
param_combinations = list(ParameterGrid(param_grid))
np.random.shuffle(param_combinations)

# Select combinations including baseline
selected_combinations = param_combinations[:max_search_combinations]

# Ensure baseline configuration is included at position 0
baseline_config = {
    'optimizer': 'rmsprop',
    'learning_rate': 0.001,
    'batch_size': 32,
    'dropout_rate': 0.5,
    'l2_reg': 1e-5,
    'dense_units_1': 256,
    'dense_units_2': 64
}

selected_combinations[0] = baseline_config  # Ensure baseline is first with detailed logging

print(f"\nStarting hyperparameter optimization...")
print(f"Estimated total time: 3-4 hours")

# Store results
optimization_results = []
start_time = time.time()

for i, params in enumerate(selected_combinations):
    combination_start = time.time()
    
    # Determine logging level
    detailed_logging = i < detailed_combinations
    
    if detailed_logging:
        print(f"\n{'='*80}")
        print(f"COMBINATION {i+1}/{len(selected_combinations)} [DETAILED LOGGING]")
        print(f"{'='*80}")
    else:
        print(f"\n{'='*60}")
        print(f"COMBINATION {i+1}/{len(selected_combinations)} [SUMMARY LOGGING]")
        print(f"{'='*60}")
    
    print(f"Configuration: {params}")
    
    try:
        # Build model with current parameters
        test_model = build_optimized_cnn_model(
            embedding_matrix,
            dropout_rate=params['dropout_rate'],
            l2_reg=params['l2_reg'],
            dense_units_1=params['dense_units_1'],
            dense_units_2=params['dense_units_2']
        )
        
        # Configure optimizer
        if params['optimizer'] == 'adam':
            optimizer = Adam(learning_rate=params['learning_rate'])
            opt_name = f"Adam(lr={params['learning_rate']})"
        else:
            optimizer = RMSprop(learning_rate=params['learning_rate'])
            opt_name = f"RMSprop(lr={params['learning_rate']})"
        
        # Compile model
        test_model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
        )
        
        if detailed_logging:
            print(f"Model compiled: {test_model.count_params():,} parameters")
            print(f"Optimizer: {opt_name}")
            print(f"Batch size: {params['batch_size']}")
            print(f"Architecture: Dropout={params['dropout_rate']}, L2={params['l2_reg']}")
            print(f"Dense layers: {params['dense_units_1']} -> {params['dense_units_2']}")
            print(f"\nStarting training with detailed epoch logging...")
            print(f"{'-'*80}")
        else:
            print(f"Model: {test_model.count_params():,} params | {opt_name} | Batch: {params['batch_size']}")
            print("Training... (summary mode)")
        
        # Train with selective verbose logging
        verbose_level = 1 if detailed_logging else 0
        history = test_model.fit(
            x=train_inputs,
            y=y_train,
            batch_size=params['batch_size'],
            epochs=15,
            validation_data=(val_inputs, y_val),
            class_weight=class_weights,
            verbose=verbose_level
        )
        
        # Extract best metrics
        best_val_acc = max(history.history['val_accuracy'])
        best_val_loss = min(history.history['val_loss'])
        best_val_prec = max(history.history['val_precision'])
        best_val_rec = max(history.history['val_recall'])
        
        # Find epoch with best validation accuracy
        best_acc_epoch = history.history['val_accuracy'].index(best_val_acc) + 1
        best_loss_epoch = history.history['val_loss'].index(best_val_loss) + 1
        
        # Calculate F1 score
        f1_scores = [2 * (p * r) / (p + r) for p, r in 
                    zip(history.history['val_precision'], history.history['val_recall'])]
        best_f1 = max(f1_scores)
        
        # Compare with baseline
        acc_diff = best_val_acc - baseline_val_acc
        loss_diff = baseline_val_loss - best_val_loss
        
        if detailed_logging:
            print(f"{'-'*80}")
            print(f"TRAINING COMPLETED - COMBINATION {i+1} SUMMARY:")
            print(f"{'='*60}")
            print(f"Best Validation Metrics:")
            print(f"  Val Accuracy: {best_val_acc:.4f} (epoch {best_acc_epoch}) | Precision: {best_val_prec:.4f} | Recall: {best_val_rec:.4f} | F1: {best_f1:.4f}")
            print(f"  Val Loss: {best_val_loss:.4f} (epoch {best_loss_epoch})")
            print(f"Comparison with Baseline:")
            print(f"  Accuracy: {acc_diff:+.4f} ({acc_diff*100:+.3f}%)")
            print(f"  Loss: {loss_diff:+.4f} ({loss_diff*100:+.3f}% improvement)")
            if acc_diff > 0:
                print(f"  >>> IMPROVEMENT FOUND! <<<")
            else:
                print(f"  No improvement over baseline")
        else:
            # Summary logging
            improvement_status = "IMPROVED" if acc_diff > 0 else "baseline"
            print(f"Results: Val Acc {best_val_acc:.4f} | Val Loss {best_val_loss:.4f} | F1 {best_f1:.4f}")
            print(f"vs Baseline: {acc_diff*100:+.3f}% acc, {loss_diff*100:+.3f}% loss | Status: {improvement_status}")
        
        # Store results
        result = {
            'combination': i+1,
            'parameters': params,
            'best_val_accuracy': best_val_acc,
            'best_val_loss': best_val_loss,
            'best_val_precision': best_val_prec,
            'best_val_recall': best_val_rec,
            'best_f1_score': best_f1,
            'best_acc_epoch': best_acc_epoch,
            'best_loss_epoch': best_loss_epoch,
            'model_parameters': test_model.count_params(),
            'epochs_trained': len(history.history['val_accuracy']),
            'baseline_improvement': acc_diff,
            'training_time_seconds': time.time() - combination_start
        }
        
        optimization_results.append(result)
        
        elapsed_time = time.time() - start_time
        estimated_remaining = (elapsed_time / (i + 1)) * (len(selected_combinations) - i - 1)
        print(f"Time: {elapsed_time//60:.0f}m {elapsed_time%60:.0f}s elapsed | Est. remaining: {estimated_remaining//60:.0f}m {estimated_remaining%60:.0f}s")
        
    except Exception as e:
        print(f"ERROR in combination {i+1}: {str(e)}")
        continue

# Sort results by validation accuracy
optimization_results.sort(key=lambda x: x['best_val_accuracy'], reverse=True)

total_time = time.time() - start_time
print(f"\n{'='*80}")
print("HYPERPARAMETER OPTIMIZATION COMPLETED")
print(f"{'='*80}")
print(f"Total optimization time: {total_time//60:.0f}m {total_time%60:.0f}s")
print(f"Successful combinations tested: {len(optimization_results)}")

if optimization_results:
    print(f"\nTOP 10 CONFIGURATIONS RANKED BY VALIDATION ACCURACY:")
    print(f"{'='*80}")
    for i, result in enumerate(optimization_results[:10]):
        print(f"Rank {i+1:2d}: Acc {result['best_val_accuracy']:.4f} | Loss {result['best_val_loss']:.4f} | F1 {result['best_f1_score']:.4f}")
        print(f"         {result['parameters']}")
        print(f"         Improvement: {result['baseline_improvement']*100:+.3f}% | Time: {result['training_time_seconds']//60:.0f}m")
        print()

print("Task 3 completed successfully")


Task 3: Hyperparameter Grid Search Optimization
Hyperparameter search space:
  optimizer: ['adam', 'rmsprop']
  learning_rate: [0.0005, 0.001, 0.002]
  batch_size: [16, 32, 64]
  dropout_rate: [0.3, 0.5, 0.7]
  l2_reg: [1e-06, 1e-05, 0.0001]
  dense_units_1: [128, 256, 512]
  dense_units_2: [32, 64, 128]

Total possible combinations: 1,458
Testing 25 strategic combinations
Detailed epoch logging for first 5 combinations
Summary logging for remaining 20 combinations

Starting hyperparameter optimization...
Estimated total time: 3-4 hours

COMBINATION 1/25 [DETAILED LOGGING]
Configuration: {'optimizer': 'rmsprop', 'learning_rate': 0.001, 'batch_size': 32, 'dropout_rate': 0.5, 'l2_reg': 1e-05, 'dense_units_1': 256, 'dense_units_2': 64}
Model compiled: 6,859,409 parameters
Optimizer: RMSprop(lr=0.001)
Batch size: 32
Architecture: Dropout=0.5, L2=1e-05
Dense layers: 256 -> 64

Starting training with detailed epoch logging...
------------------------------------------------------------------

In [ ]:
print("Task 4: Hyperparameter Optimization Results Analysis")
print("=" * 60)

# Check if optimization_results exists from previous task
if 'optimization_results' not in locals() or len(optimization_results) == 0:
    print("ERROR: No optimization results found. Please run Task 3 first.")
    print("Task 4 cannot proceed without optimization_results data.")
else:
    print(f"Found {len(optimization_results)} optimization results for analysis")
    
    # Convert dynamic optimization_results to DataFrame
    results_data = []
    for i, result in enumerate(optimization_results):
        results_data.append({
            'rank': i + 1,
            'accuracy': result['best_val_accuracy'],
            'loss': result['best_val_loss'],
            'f1': result['best_f1_score'],
            'improvement': result['baseline_improvement'] * 100,  # Convert to percentage
            'batch_size': result['parameters']['batch_size'],
            'dropout': result['parameters']['dropout_rate'],
            'lr': result['parameters']['learning_rate'],
            'optimizer': result['parameters']['optimizer'],
            'dense1': result['parameters']['dense_units_1'],
            'dense2': result['parameters']['dense_units_2'],
            'l2_reg': result['parameters']['l2_reg'],
            'training_time': result['training_time_seconds'],
            'epochs': result['epochs_trained'],
            'model_params': result['model_parameters']
        })
    
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    
    # Convert to DataFrame
    df = pd.DataFrame(results_data)
    
    print(f"Analysis data prepared:")
    print(f"  Top accuracy: {df['accuracy'].max():.4f}")
    print(f"  Best improvement: +{df['improvement'].max():.3f}%")
    print(f"  Configurations analyzed: {len(df)}")
    
    # Dynamic baseline values (from your Phase 5 results)
    baseline_acc = baseline_val_acc  # This should be available from Task 1
    baseline_loss = baseline_val_loss
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Performance Ranking (Dynamic)
    colors = ['green' if x >= baseline_acc else 'red' for x in df['accuracy']]
    bars = axes[0,0].barh(range(len(df[:10])), df['accuracy'][:10], color=colors[:10])
    axes[0,0].axvline(baseline_acc, color='blue', linestyle='--', label=f'Baseline ({baseline_acc:.4f})')
    axes[0,0].set_yticks(range(len(df[:10])))
    axes[0,0].set_yticklabels([f'Rank {r}' for r in df['rank'][:10]])
    axes[0,0].set_xlabel('Validation Accuracy')
    axes[0,0].set_title('Top 10 Hyperparameter Configurations')
    axes[0,0].legend()
    
    # Add accuracy values on bars
    for i, bar in enumerate(bars):
        width = bar.get_width()
        axes[0,0].text(width + 0.0001, bar.get_y() + bar.get_height()/2, 
                      f'{width:.4f}', ha='left', va='center', fontsize=8)
    
    # 2. Improvement Distribution (Dynamic)
    colors = ['green' if x > 0 else 'red' if x < 0 else 'gray' for x in df['improvement']]
    axes[0,1].bar(range(len(df[:10])), df['improvement'][:10], color=colors[:10], alpha=0.7)
    axes[0,1].axhline(0, color='black', linestyle='-', linewidth=1)
    axes[0,1].set_xticks(range(len(df[:10])))
    axes[0,1].set_xticklabels([f'R{r}' for r in df['rank'][:10]], rotation=45)
    axes[0,1].set_ylabel('Improvement over Baseline (%)')
    axes[0,1].set_title('Performance Improvement Distribution')
    
    # 3. Hyperparameter Impact - Dropout Rate (Dynamic)
    dropout_effect = df.groupby('dropout').agg({
        'accuracy': ['mean', 'count'], 
        'improvement': 'mean'
    }).round(4)
    dropout_means = dropout_effect['accuracy']['mean']
    dropout_counts = dropout_effect['accuracy']['count']
    
    bars = axes[0,2].bar(dropout_means.index, dropout_means.values, 
                        alpha=0.7, color='skyblue')
    axes[0,2].axhline(baseline_acc, color='red', linestyle='--', label='Baseline')
    axes[0,2].set_xlabel('Dropout Rate')
    axes[0,2].set_ylabel('Average Validation Accuracy')
    axes[0,2].set_title('Dropout Rate Impact on Performance')
    axes[0,2].legend()
    
    # Add count labels on bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        count = dropout_counts.iloc[i]
        axes[0,2].text(bar.get_x() + bar.get_width()/2, height + 0.0001, 
                      f'n={count}', ha='center', va='bottom', fontsize=8)
    
    # 4. Batch Size vs Performance (Dynamic)
    scatter = axes[1,0].scatter(df['batch_size'], df['accuracy'], 
                               c=df['improvement'], cmap='RdYlGn', s=100, alpha=0.7)
    axes[1,0].axhline(baseline_acc, color='blue', linestyle='--', label='Baseline')
    axes[1,0].set_xlabel('Batch Size')
    axes[1,0].set_ylabel('Validation Accuracy')
    axes[1,0].set_title('Batch Size vs Performance')
    axes[1,0].legend()
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=axes[1,0])
    cbar.set_label('Improvement (%)')
    
    # 5. Learning Rate Analysis (Dynamic)
    lr_effect = df.groupby('lr').agg({
        'accuracy': ['mean', 'count'], 
        'improvement': 'mean'
    }).round(4)
    lr_means = lr_effect['accuracy']['mean']
    lr_counts = lr_effect['accuracy']['count']
    
    bars = axes[1,1].bar(lr_means.index, lr_means.values, alpha=0.7, color='lightgreen')
    axes[1,1].axhline(baseline_acc, color='red', linestyle='--', label='Baseline')
    axes[1,1].set_xlabel('Learning Rate')
    axes[1,1].set_ylabel('Average Validation Accuracy')
    axes[1,1].set_title('Learning Rate Impact on Performance')
    axes[1,1].legend()
    
    # Add count labels
    for i, bar in enumerate(bars):
        height = bar.get_height()
        count = lr_counts.iloc[i]
        axes[1,1].text(bar.get_x() + bar.get_width()/2, height + 0.0001, 
                      f'n={count}', ha='center', va='bottom', fontsize=8)
    
    # 6. Best Configuration Summary (Dynamic)
    best_config = df.iloc[0]
    config_labels = ['Batch Size', 'Dropout', 'LR (×1000)', 'Dense1', 'Dense2']
    config_values = [
        best_config['batch_size'], 
        best_config['dropout'], 
        best_config['lr'] * 1000,
        best_config['dense1'],
        best_config['dense2']
    ]
    
    # Baseline values (these should be available from your setup)
    baseline_values = [32, 0.5, 1.0, 256, 64]  # From Phase 5 baseline
    
    x = np.arange(len(config_labels))
    width = 0.35
    
    axes[1,2].bar(x - width/2, baseline_values, width, label='Baseline', alpha=0.7, color='orange')
    axes[1,2].bar(x + width/2, config_values, width, label='Best Config', alpha=0.7, color='green')
    axes[1,2].set_xlabel('Hyperparameters')
    axes[1,2].set_ylabel('Values')
    axes[1,2].set_title('Best Configuration vs Baseline')
    axes[1,2].set_xticks(x)
    axes[1,2].set_xticklabels(config_labels, rotation=45)
    axes[1,2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Dynamic detailed analysis
    print("\nHYPERPARAMETER OPTIMIZATION ANALYSIS")
    print("=" * 50)
    
    best_result = df.iloc[0]
    print(f"Best Model Performance:")
    print(f"  Validation Accuracy: {best_result['accuracy']:.4f} (+{best_result['improvement']:.3f}%)")
    print(f"  F1 Score: {best_result['f1']:.4f}")
    print(f"  Validation Loss: {best_result['loss']:.4f}")
    print(f"  Model Parameters: {best_result['model_params']:,}")
    print(f"  Training Time: {best_result['training_time']//60:.0f}m {best_result['training_time']%60:.0f}s")
    
    print(f"\nOptimal Hyperparameter Configuration:")
    print(f"  Optimizer: {best_result['optimizer']}")
    print(f"  Learning Rate: {best_result['lr']:.4f}")
    print(f"  Batch Size: {best_result['batch_size']}")
    print(f"  Dropout Rate: {best_result['dropout']}")
    print(f"  Dense Units: {best_result['dense1']} -> {best_result['dense2']}")
    print(f"  L2 Regularization: {best_result['l2_reg']}")
    
    # Dynamic insights based on actual data
    top_configs = df.head(5)
    common_dropout = top_configs['dropout'].mode().iloc[0]
    common_optimizer = top_configs['optimizer'].mode().iloc[0]
    avg_improvement = df[df['improvement'] > 0]['improvement'].mean()
    
    print(f"\nKey Insights from {len(df)} Configurations:")
    print(f"  - Most effective dropout rate: {common_dropout}")
    print(f"  - Best optimizer: {common_optimizer}")
    print(f"  - Average improvement in successful configs: +{avg_improvement:.3f}%")
    print(f"  - Configurations showing improvement: {len(df[df['improvement'] > 0])}/{len(df)}")
    
    # Dynamic production impact
    baseline_errors = int((1 - baseline_acc) * 12000)
    optimized_errors = int((1 - best_result['accuracy']) * 12000)
    error_reduction = baseline_errors - optimized_errors
    
    print(f"\nProduction Impact:")
    print(f"  Baseline errors per 12K samples: {baseline_errors}")
    print(f"  Optimized errors per 12K samples: {optimized_errors}")
    print(f"  Error reduction: {error_reduction} samples ({error_reduction/baseline_errors*100:.1f}%)")
    
    print("\nTask 4 completed successfully")
